
# Acquire bead scatter on the FACSCanto

Add a bead population to the FACSCanto preset, dilute the sample and
inspect its forward- and side-scatter voltage traces.


## Step 0: Global Settings and Imports



In [ ]:
from FlowCyPy.units import ureg
from FlowCyPy.systems.bd import facs_canto
from FlowCyPy.fluidics import SampleFlowRate, SheathFlowRate
from FlowCyPy.fluidics import populations, distributions

## Define the sample
Use a narrow particle-size distribution to inspect the preset instrument response.



In [ ]:
diameter = distributions.RosinRammler(
    scale=60 * ureg.nanometer,
    shape=150,
)

medium_refractive_index = 1.33

refractive_index = distributions.Normal(mean=1.44, standard_deviation=0.002)

population_0 = populations.SpherePopulation(
    name="Pop 0",
    concentration=5e9 * ureg.particle / ureg.milliliter,
    diameter=diameter,
    medium_refractive_index=medium_refractive_index,
    refractive_index=refractive_index,
)

## Configure the instrument
Choose flow rates, laser power, noise sources and the trigger threshold.



In [ ]:
instrument = facs_canto(
    sample_volume_flow=SampleFlowRate.MEDIUM,
    sheath_volume_flow=SheathFlowRate.DEFAULT,
    optical_power=200 * ureg.milliwatt,
    threshold="3sigma",
    include_shot_noise=True,
    include_rin_noise=True,
    background_power=0.01 * ureg.nanowatt,
)

sample = instrument.fluidics.scatterer_collection
sample.add_population(population_0)

sample.dilute(factor=100)

## Acquire the sample
Dilute the population and simulate a short acquisition.



In [ ]:
run_record = instrument.run(run_time=0.2 * ureg.millisecond)

## Inspect the analog signal
Look for individual pulses and the noise baseline in each detector channel.



In [ ]:
run_record.plot_analog(include_scatterer=False)